<a href="https://colab.research.google.com/github/shin584/project/blob/3D_simulation/nucleotide_model_test.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
# [Cell 1] multimolecule 미러 모델 호환을 위한 최신 Transformers 세팅
!pip install -q --upgrade transformers datasets accelerate evaluate scikit-learn scipy einops

import torch
import transformers
print("=== 환경 체크 ===")
print(f"PyTorch Version      : {torch.__version__}")
print(f"Transformers Version : {transformers.__version__} (최신 버전)")
print(f"GPU Available        : {torch.cuda.is_available()}")

=== 환경 체크 ===
PyTorch Version      : 2.11.0+cu128
Transformers Version : 5.14.1 (최신 버전)
GPU Available        : True


In [3]:
# [Cell 2] Google DeepMind/InstaDeep의 표준 DNA 파운데이션 모델 토큰화
import os
import pandas as pd
from datasets import load_dataset
from transformers import AutoTokenizer

print("=== [Phase 3.2] 데이터셋 로드 및 토큰화 (Nucleotide Transformer) ===")

required_files = ["train.csv", "val.csv", "test.csv"]
for file_name in required_files:
    if not os.path.exists(file_name):
        raise FileNotFoundError(f"'{file_name}' 파일을 찾을 수 없습니다.")

raw_datasets = load_dataset("csv", data_files={
    "train": "train.csv",
    "validation": "val.csv",
    "test": "test.csv"
})

sample_df = pd.read_csv("train.csv", nrows=1)
seq_col = next((c for c in sample_df.columns if c.lower() in ["input_sequence", "sequence", "target_sequence", "seq"]), sample_df.columns[0])
label_col = next((c for c in sample_df.columns if c.lower() in ["score", "label", "efficiency", "cleavage_efficiency", "cleavage_score"]), sample_df.columns[1])
print(f"\n매핑된 컬럼명 -> 서열: '{seq_col}' | 라벨: '{label_col}'")

# trust_remote_code=True가 전혀 필요 없는 100% Hugging Face 정식 표준 모델!
model_name = "InstaDeepAI/nucleotide-transformer-500m-human-ref"
print(f"\n'{model_name}' 토크나이저 로드 중...")
tokenizer = AutoTokenizer.from_pretrained(model_name)

def preprocess_function(examples):
    tokenized = tokenizer(
        examples[seq_col],
        padding="max_length",
        truncation=True,
        max_length=64
    )
    tokenized["label"] = [float(val) for val in examples[label_col]]
    return tokenized

print("전체 데이터셋 토큰화 적용 중...")
tokenized_datasets = raw_datasets.map(
    preprocess_function,
    batched=True,
    remove_columns=raw_datasets["train"].column_names
)
print("\n토큰화 완료!")

=== [Phase 3.2] 데이터셋 로드 및 토큰화 (Nucleotide Transformer) ===

매핑된 컬럼명 -> 서열: 'input_sequence' | 라벨: 'score'

'InstaDeepAI/nucleotide-transformer-500m-human-ref' 토크나이저 로드 중...


config.json:   0%|          | 0.00/706 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/129 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/28.7k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/101 [00:00<?, ?B/s]

전체 데이터셋 토큰화 적용 중...


Map:   0%|          | 0/4116 [00:00<?, ? examples/s]

Map:   0%|          | 0/514 [00:00<?, ? examples/s]

Map:   0%|          | 0/515 [00:00<?, ? examples/s]


토큰화 완료!


In [4]:
# [Cell 3] Nucleotide Transformer 회귀 헤드 안전 로딩 및 순전파 검증
import torch
from transformers import AutoModelForSequenceClassification

print("=== [Phase 3.3] 사전 학습 모델 및 회귀 헤드 로드 (표준 Native 모델) ===")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_name = "InstaDeepAI/nucleotide-transformer-500m-human-ref"

# Hugging Face 공식 아키텍처이므로 커스텀 충돌 없이 회귀 모델(num_labels=1) 부착
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=1,
    ignore_mismatched_sizes=True
)

model.to(device)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"\n모델 로드 성공: '{model_name}' (100% Hugging Face Native Architecture)")
print(f"Total Parameters     : {total_params:,}")
print(f"Trainable Parameters : {trainable_params:,}")

# 샘플 텐서 순전파(Forward Pass) 검증
test_batch = {
    "input_ids": torch.tensor([tokenized_datasets["train"][0]["input_ids"]]).to(device),
    "attention_mask": torch.tensor([tokenized_datasets["train"][0]["attention_mask"]]).to(device),
    "labels": torch.tensor([tokenized_datasets["train"][0]["label"]], dtype=torch.float32).to(device)
}

model.eval()
with torch.no_grad():
    sample_output = model(**test_batch)

print("\n[샘플 텐서 순전파 테스트]")
print(f"입력 input_ids shape   : {test_batch['input_ids'].shape}")
print(f"출력 logits shape      : {sample_output.logits.shape}  --> (Batch=1, Num_Labels=1)")
print(f"초기 미학습 손실(Loss) : {sample_output.loss.item():.4f}")
print("-" * 50)
print("3단계 환경 세팅 및 샘플 검증이 완벽히 끝났습니다!")

=== [Phase 3.3] 사전 학습 모델 및 회귀 헤드 로드 (표준 Native 모델) ===


pytorch_model.bin: reconstructing file:   0%|          |  0.00B / 1.94GB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/390 [00:00<?, ?it/s]

[transformers] EsmForSequenceClassification LOAD REPORT from: InstaDeepAI/nucleotide-transformer-500m-human-ref
Key                        | Status     | 
---------------------------+------------+-
lm_head.decoder.weight     | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
classifier.out_proj.weight | MISSING    | 
classifier.dense.weight    | MISSING    | 
classifier.out_proj.bias   | MISSING    | 
classifier.dense.bias      | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



모델 로드 성공: 'InstaDeepAI/nucleotide-transformer-500m-human-ref' (100% Hugging Face Native Architecture)
Total Parameters     : 480,439,522
Trainable Parameters : 480,439,522


model.safetensors: reconstructing file:   0%|          |  0.00B / 1.94GB            

model.safetensors: downloading bytes:           |  0.00B            


[샘플 텐서 순전파 테스트]
입력 input_ids shape   : torch.Size([1, 64])
출력 logits shape      : torch.Size([1, 1])  --> (Batch=1, Num_Labels=1)
초기 미학습 손실(Loss) : 0.7870
--------------------------------------------------
3단계 환경 세팅 및 샘플 검증이 완벽히 끝났습니다!
